# Sprint 3: E1 và E8 (pha A, pha B)

Cấu hình Đăng xác nhận: **gamma=1.0 + L2**, D=25.000 cho E1, lambda theo paper; RFF/thống kê/giải float64.
Paper không mô tả cụ thể chuẩn hóa; L2 là quyết định của nhóm. E8-B có D=5.000/25.000, ba cách shrink, lambda cố định và tương đối.

**Trước khi chạy:** bật GPU T4, Internet và Secret `WANDB_API_KEY` (quyền ghi project chung). `HF_TOKEN` là tùy chọn. Code cần được push lên nhánh ghi ở ô kế tiếp.
**Kết quả:** 15 run E1 + 32 run E8-A (chuẩn hóa × γ) + 24 run E8-B (shrink × λ), mỗi run một tiến trình và một URL W&B. Thời gian toàn bộ chưa đo. Có thể lọc theo dataset/seed để chia phiên.
**Tiếp tục:** giữ thư mục `/kaggle/working/sprint3-results`; sau mỗi phiên bấm Save Version và lưu outputs. Phiên mới cần phục hồi các JSON đã lưu vào cùng thư mục. Chỉ skip run khi source, cấu hình và URL W&B khớp; không sửa source giữa các phiên trong cùng bộ kết quả.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/24122039Kien/RanDumb-reproduction.git'
REPO_BRANCH = 'main'  # nhánh của Đăng đã merge vào main ngày 19/11
REPO_DIR = Path('/kaggle/working/RanDumb-reproduction')
OUTPUT_DIR = Path('/kaggle/working/sprint3-results')
GROUP = 'all'       # 'E1', 'E8-A', 'E8-B', 'E8' (cả A và B), hoặc 'all'
DATASET = None      # ví dụ 'mnist' để chia phiên; None chạy tất cả
SEED = None         # E1: 0/1/2; E8: chỉ seed 0
DIM = None          # E8: 5000 hoặc 25000 để chia phiên; None chạy cả hai
BATCH_SIZE = 512


In [ ]:
if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Repo có thay đổi chưa commit; lưu chúng trước khi cập nhật.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
try:
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
except Exception:
    pass
os.environ['RANDUMB_DATA_DIR'] = '/kaggle/working/data'
os.environ['PYTHONUNBUFFERED'] = '1'
# Giảm phân mảnh VRAM khi cấp phát các ma trận D×D 4,66 GiB (D = 25.000, float64)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Không in biến môi trường hoặc khóa. Kiểm tra code trước khi chạy dữ liệu thật.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)


## Kiểm tra bộ nhớ ở đúng D=25.000
Đây là ảnh giả để phát hiện OOM trước khi tải dataset, không phải kết quả E1/E8. Nếu lỗi thì dừng và xem JSON/log; không tự giảm D hoặc đổi dtype. Peak của dữ liệu thật vẫn cần đo ở từng run.


In [ ]:
for shrink in ['oas', 'oas_per_class']:
    subprocess.run([sys.executable, str(REPO_DIR / 'scripts/measure_sprint3_memory.py'),
                    '--dim', '25000', '--shrink', shrink,
                    '--output', str(OUTPUT_DIR / f'memory-D25000-{shrink}.json')],
                   cwd=REPO_DIR, check=True)


In [ ]:
command = [sys.executable, str(REPO_DIR / 'scripts/run_sprint3.py'),
           '--group', GROUP, '--device', 'cuda', '--batch-size', str(BATCH_SIZE),
           '--out-dir', str(OUTPUT_DIR)]
if DATASET is not None:
    command += ['--dataset', DATASET]
if SEED is not None:
    command += ['--seed', str(SEED)]
if DIM is not None:
    command += ['--dim', str(DIM)]
subprocess.run(command, cwd=REPO_DIR, check=True)              # xem manifest
subprocess.run(command + ['--execute'], cwd=REPO_DIR, check=True)


In [ ]:
import csv
for filename in ['e1_aggregate.csv', 'summary.csv']:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(filename)
        with path.open(encoding='utf-8', newline='') as stream:
            for row in csv.DictReader(stream):
                print(row)
print('Save Version và giữ toàn bộ JSON/CSV trong:', OUTPUT_DIR)
